# 공통 Neo4j 적재와 실제 키 검증 (B06·B07)

`school_normalize_bundle.ipynb`가 만든 적재 패키지(`bundle.json`)를 **공통 Neo4j**에 넣는다.

1. 스키마: 온톨로지 v3.1 5-3절의 고유 제약 6개 + Chunk 전문 검색(cjk) + stage·scope 인덱스 + 벡터 인덱스(팀 합의 `text-embedding-3-small`, 1536차원, cosine). `IF NOT EXISTS`는 기존 인덱스 설정을 바꾸지 않으므로 실제 설정(분석기·차원)을 다시 읽어 확인한다.
2. 표준 사전 노드(Disease 11·SchoolRole 11·ActionConcept 27) MERGE.
3. 적재 → 담당 문서 범위의 실제 키·속성 대조 → **같은 입력 재적재** 후 키 집합 비교.
4. 끝점 누락(없는 청크로 가는 관계)이 조용히 사라지지 않고 트랜잭션 전체가 되돌려지는지 확인.
5. Chunk 임베딩(바뀐 청크만) → 6. 대표 질문 경로(청크 → 행동 → 역할) 읽기 조회.

모든 쓰기는 MERGE·IF NOT EXISTS이며 삭제·변경 문장은 없다. 접속 정보는 `.env`에서만 읽고 출력하지 않는다. 결과는 `data/interim/school/load_ledger.jsonl`에 남는다.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from index.school.dictionaries import load_dictionaries
from index.school.embed import EMBEDDING_MODEL, embed_chunks, make_embedder
from index.school.load import (
    LoadError,
    fetch_scope_keys,
    is_clean,
    load_bundle,
    load_with_retry,
    verify_loaded_bundle,
)
from index.school.neo4j_local import get_driver
from index.school.schema_init import apply_schema, missing_dictionary_refs, schema_state, seed_dictionaries

BUNDLE_DIR = ROOT / "data" / "interim" / "school" / "full_20261104T075351_804218Z"
LEDGER = ROOT / "data" / "interim" / "school" / "load_ledger.jsonl"
bundle = json.loads((BUNDLE_DIR / "bundle.json").read_text(encoding="utf-8"))
bundle["dictionary_refs"] = [tuple(x) for x in bundle["dictionary_refs"]]
print("패키지:", BUNDLE_DIR.name, {k: len(v) for k, v in bundle["nodes"].items()}, "관계", len(bundle["edges"]))

driver, DATABASE = get_driver(ROOT / ".env")
print("Neo4j 연결 확인")

패키지: full_20261104T075351_804218Z {'Document': 2, 'Chunk': 182, 'SchoolAction': 42} 관계 316


Neo4j 연결 확인


## 1. 스키마(제약·인덱스)

In [2]:
before = schema_state(driver, DATABASE)
print("적용 전 빠진 고유 제약:", before["missing_constraints"])
apply_schema(driver, DATABASE)
after = schema_state(driver, DATABASE)
print("적용 후 빠진 고유 제약:", after["missing_constraints"] or "없음",
      "/ 빠진 인덱스:", after["missing_indexes"] or "없음", "/ 설정이 다른 인덱스:", after["wrong_index_options"] or "없음")
display(pd.DataFrame([{"index": k, "type": v[0], "state": v[1]} for k, v in sorted(after["indexes"].items())]))
assert not after["missing_constraints"] and not after["missing_indexes"] and not after["wrong_index_options"]

적용 전 빠진 고유 제약: []


적용 후 빠진 고유 제약: 없음 / 빠진 인덱스: 없음 / 설정이 다른 인덱스: 없음


,index,type,state
0,action_id,RANGE,ONLINE
1,action_scope,RANGE,ONLINE
2,action_stage,RANGE,ONLINE
3,chunk_embedding,VECTOR,ONLINE
4,chunk_id,RANGE,ONLINE
5,chunk_text,FULLTEXT,ONLINE
6,concept_id,RANGE,ONLINE
7,disease_id,RANGE,ONLINE
8,document_id,RANGE,ONLINE
9,index_1b9dcc97,LOOKUP,ONLINE


## 2. 표준 사전 노드

In [3]:
books = load_dictionaries()
print("사전 MERGE:", seed_dictionaries(driver, books, DATABASE))
missing = missing_dictionary_refs(driver, bundle["dictionary_refs"], DATABASE)
print("패키지가 연결할 사전 노드 중 DB에 없는 것:", missing or "없음")
assert not missing

사전 MERGE: {'Disease': 11, 'SchoolRole': 11, 'ActionConcept': 27}


패키지가 연결할 사전 노드 중 DB에 없는 것: 없음


## 3. 적재와 실제 키 검증

적재 카운터가 아니라 DB를 다시 읽어 기대 키·속성과 비교한다. 비교 범위는 school 담당 문서 2개와 그 ID로 시작하는 청크·행동, 거기서 나가는 관계다(팀원 데이터 건수와 섞지 않음).

In [4]:
first = load_with_retry(driver, bundle, ledger=LEDGER, bundle_dir=BUNDLE_DIR.name, database=DATABASE)
print("상태:", first["status"])
print("처리 행 수:", first["written"])
print({k: (len(v) if isinstance(v, list) else v) for k, v in first["report"].items()})
assert is_clean(first["report"])
keys_after_first = fetch_scope_keys(driver, bundle, DATABASE)

상태: SUCCESS
처리 행 수: {'Document': 2, 'Chunk': 182, 'HAS_CHUNK': 182, 'ABOUT': 31, 'SchoolAction': 42, 'SUPPORTED_BY': 42, 'DONE_BY': 42, 'INSTANCE_OF': 19}
{'missing_nodes': 0, 'extra_nodes': 0, 'property_mismatch': 0, 'missing_edges': 0, 'extra_edges': 0, 'actual_counts': {'nodes': 226, 'edges': 316}}


## 4. 같은 입력 재적재 - 키 집합이 그대로인지

In [5]:
second = load_with_retry(driver, bundle, ledger=LEDGER, bundle_dir=BUNDLE_DIR.name, database=DATABASE)
keys_after_second = fetch_scope_keys(driver, bundle, DATABASE)
same_nodes = keys_after_first["nodes"] == keys_after_second["nodes"]
same_edges = keys_after_first["edges"] == keys_after_second["edges"]
print("재적재 상태:", second["status"], "/ 노드 키·속성 동일:", same_nodes, "/ 관계 키 동일:", same_edges)
print("관계 수:", len(keys_after_second["edges"]), "(중복 관계가 생기면 늘어난다)")
assert same_nodes and same_edges and is_clean(second["report"])

재적재 상태: SUCCESS / 노드 키·속성 동일: True / 관계 키 동일: True
관계 수: 316 (중복 관계가 생기면 늘어난다)


## 5. 끝점 누락 검출 - 조용히 사라지지 않고 전체가 되돌려지는지

없는 청크(`…p0036:t09r09`)로 가는 `SUPPORTED_BY`를 하나 섞은 패키지를 넣어 본다. MATCH가 그 행을 버리면 처리 행 수가 줄어 `LoadError`가 나고, 행동 트랜잭션 전체가 되돌려져야 한다. 이후 검증이 여전히 깨끗해야 한다.

In [6]:
broken = json.loads(json.dumps(bundle))
action_id = broken["nodes"]["SchoolAction"][0]["action_id"]
broken["edges"].append({"type": "SUPPORTED_BY", "start_label": "SchoolAction", "start_id": action_id,
                        "end_label": "Chunk", "end_id": "moe_school_manual_v3:p0036:t09r09", "props": {}})
try:
    load_bundle(driver, broken, DATABASE)
    raise AssertionError("끝점 누락이 검출되지 않음")
except LoadError as exc:
    print("검출:", exc)
report = verify_loaded_bundle(driver, bundle, DATABASE)
print("되돌린 뒤 검증 깨끗함:", is_clean(report))
assert is_clean(report)

검출: SUPPORTED_BY: 입력 43행 중 42행만 처리됨 - 끝점 누락


되돌린 뒤 검증 깨끗함: True


## 6. Chunk 임베딩

Chunk.text를 팀 합의 모델로 임베딩해 `embedding`에 넣는다. 텍스트·모델이 장부(`embed_ledger.jsonl`)와 같고 DB에 벡터가 있으면 건너뛴다(재실행 비용 없음).

In [7]:
embedded = embed_chunks(driver, bundle["nodes"]["Chunk"], make_embedder(),
                        ledger=ROOT / "data" / "interim" / "school" / "embed_ledger.jsonl", database=DATABASE)
print("모델:", EMBEDDING_MODEL, "/ 새로 임베딩:", len(embedded["embedded"]), "/ 건너뜀:", len(embedded["skipped"]))
print("임베딩 후 검증 깨끗함:", is_clean(verify_loaded_bundle(driver, bundle, DATABASE)))

모델: text-embedding-3-small / 새로 임베딩: 0 / 건너뜀: 182


임베딩 후 검증 깨끗함: True


## 7. 대표 질문 경로 미리 보기 (읽기 전용)

'유증상 학생 발견 시 담임·보건교사가 확인할 일'(S-Q1)에 쓰일 경로: 대응 1단계 행동 → 근거 청크 → 문서, 수행 역할과 basis. 실제 검색기(B08)는 Chunk 검색을 먼저 하지만, 여기서는 적재된 경로가 이어지는지만 본다.

In [8]:
records, _, _ = driver.execute_query('''
MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)<-[:SUPPORTED_BY]-(a:SchoolAction)-[r:DONE_BY]->(role:SchoolRole)
WHERE d.document_id IN $docs AND a.stage = 'RESPONSE_1' AND role.role_id IN ['HOMEROOM_TEACHER', 'HEALTH_TEACHER']
OPTIONAL MATCH (a)-[:INSTANCE_OF]->(k:ActionConcept)
RETURN d.document_id AS document, c.pdf_page AS pdf, c.printed_page AS printed, role.role_id AS role,
       r.basis AS basis, a.step_order AS step, a.action_id AS action_id, a.action_text AS action_text,
       k.concept_id AS concept
ORDER BY document, role, step, action_id''', docs=[d["document_id"] for d in bundle["nodes"]["Document"]],
    database_=DATABASE)
paths = pd.DataFrame([r.data() for r in records])
print("경로 수:", len(paths), "/ 문서·역할별:", dict(Counter(zip(paths["document"], paths["role"]))))
print("basis:", dict(Counter(paths["basis"])), "/ 개념 연결:", int(paths["concept"].notna().sum()), "/", len(paths))
pd.set_option("display.max_colwidth", 70)
display(paths)

경로 수: 42 / 문서·역할별: {('gyeonggi_school_guide_2025', 'HEALTH_TEACHER'): 10, ('gyeonggi_school_guide_2025', 'HOMEROOM_TEACHER'): 10, ('moe_school_manual_v3', 'HEALTH_TEACHER'): 12, ('moe_school_manual_v3', 'HOMEROOM_TEACHER'): 10}
basis: {'EXAMPLE': 42} / 개념 연결: 19 / 42


,document,pdf,printed,role,basis,step,action_id,action_text,concept
0,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,1,gyeonggi_school_guide_2025:p0054:a11,교사와 진한이가 보건실에 도착한 즉시 손을 씻도록 요청한다.,NaN
1,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,2,gyeonggi_school_guide_2025:p0054:a12,마스크를 착용한다.,NaN
2,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,2,gyeonggi_school_guide_2025:p0054:a13,체온을 측정한다.,DETECT_SYMPTOMATIC
3,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,2,gyeonggi_school_guide_2025:p0054:a14,환자의 증상과 징후를 관찰하고 감염병이 의심되는지를 확인한다.,DETECT_SYMPTOMATIC
4,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,3,gyeonggi_school_guide_2025:p0054:a15,진한이가 의료기관 진료를 받지 않았음을 확인한다.,NaN
5,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,4,gyeonggi_school_guide_2025:p0054:a16,교무부장에게 연락하여 격리 담당 교사의 결정을 요청한다.,NaN
6,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,5,gyeonggi_school_guide_2025:p0054:a17,진한이를 일시적 관찰실로 이동한다.,ESCORT_TO_HEALTH_ROOM
7,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,5,gyeonggi_school_guide_2025:p0054:a18,부모님이 올 때까지 일시적 관찰실에서 진한이를 보호·관찰한다.,NaN
8,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,5,gyeonggi_school_guide_2025:p0054:a19,진한이의 상태를 주기적으로 확인한다.,NaN
9,gyeonggi_school_guide_2025,54,44,HEALTH_TEACHER,EXAMPLE,6,gyeonggi_school_guide_2025:p0054:a20,‘일시적 관찰실 출입 금지’ 안내판을 부착한다.,NaN


In [9]:
for line in LEDGER.read_text(encoding="utf-8").splitlines()[-2:]:
    entry = json.loads(line)
    print(entry["timestamp_utc"], entry["status"], entry["bundle_sha256"][:12], entry["verify"]["clean"])
driver.close()

2026-11-04T07:54:22.914660+00:00 SUCCESS 8628542f02ff True
2026-11-04T07:54:25.001658+00:00 SUCCESS 8628542f02ff True
